EM Algorithm
=

In [1]:
import sys

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [2]:
print(sys.executable)

/home/simonque/Documents/code/env/bin/python3


In [3]:
print(sys.prefix)

/home/simonque/Documents/code/env


In [4]:
debug = 0
def printd(s: str, debug:int):
    if debug: 
        print(s)

# Import Libraries

In [5]:
import numpy as np 
import matplotlib.pyplot as plt
import struct
import argparse


# Define input filenames and parameters

In [9]:
images_train_filename = "train-images.idx3-ubyte___"
labels_train_filename = "train-labels.idx1-ubyte___"

# Parse the input binary files
Load the files as numpy arrays


In [10]:
def load_mnist(images_filepath, labels_filepath):
    with open(images_filepath, "rb") as f:
        # read the first four 32-bit integers
        header = f.read(4*4)
        # read the magic number, number of images, number of rows, and number of columns 
        magic_number, num_images, rows, cols = struct.unpack(">IIII", header)
        printd(f"{magic_number}, {num_images}, {rows}, {cols}", debug)

        raw_image = f.read()

        printd(type(raw_image), debug)
        # create empty numpy array of shape 
        images = np.frombuffer(raw_image, dtype=np.uint8).reshape((num_images, rows, cols))
        printd(images.shape, debug)
    with open(labels_filepath, "rb") as f:
        # read the first two 32-bit integers
        header = f.read(2*4)
        # read the magic number, number of images, number of rows, and number of columns 
        magic_number, num_labels = struct.unpack(">II", header)
        printd(f"{magic_number}, {num_labels}", debug)

        raw_labels = f.read()

        printd(type(raw_labels), debug)
        # create empty numpy array of shape 
        labels = np.frombuffer(raw_labels, dtype=np.uint8).reshape((num_labels))
        printd(labels.shape, debug)
    return images, labels
    


## Load training dataset

In [11]:
images_train, labels_train = load_mnist(images_train_filename, labels_train_filename)
images_train.shape, labels_train.shape

((60000, 28, 28), (60000,))

## Binarize images

In [44]:
def threshold_images(images: np.array,
                   threshold: int = 127):
    return (images > threshold).astype(float)

# testing 
images_train_thresholded = threshold_images(images_train)

In [45]:
'''
vectorized Bernoulli log-likelihood formula
'''
def bernoulli_log_likelihood(X: np.array,
                             mu: np.array):
    return (X @ np.log(mu).T + (1.0 - X) @ np.log(1.0 - mu).T)

# EM algorithm

In [97]:
def train_em_algorithm(X: np.array,
                n_clusters: int = 10,
                max_iters:int = 50,
                tolerance = 1e-4, 
                smoothing = 1e-10,
                rand = 0.25):
    # --- initialize parameters ---
    # reshape dimensions from 28x28 to 784
    X = np.reshape(X, (X.shape[0], -1))
    n_samples, n_dimensions = X.shape
    print(f"X.shape = {X.shape}")
    print(f"n_samples = {n_samples}, n_dimensions = {n_dimensions}")
    
    # uniform prior weights 
    pi = np.ones(n_clusters) / n_clusters 
    print(f"pi.shape = {pi.shape}")
    
    # initialize mean values to random values
    mu = np.random.uniform(rand, 1-rand, size = (n_clusters, n_dimensions))
    print(f"mu.shape = {mu.shape}")

    prev_mu = mu
    prev_pi = pi
    
    # iteratively perform the EM algorithm
    print(f"Training")
    for i in range(max_iters):
        print(f"iteration {i}")
        # --- Expectation step ---
        # compute responsibilities
                               
        # --- compute log likelihood --- 
        # log (P(X | mu)
        # clip the weights to avoid log(0)
        mu = np.clip(mu, smoothing, 1.0 - smoothing)
        log_likelihood = bernoulli_log_likelihood(X, mu)
        print(f"log_likelihood.shape = {log_likelihood.shape}")
    
        # --- get the joint probability ---
        # add the log likelihood of the weights and prior weights
        log_joint = log_likelihood + np.log(np.clip(pi, 1e-300, 1.0))
        print(f"log_joint.shape = {log_joint.shape}")

        # --- get the marginal log likelihood ---
        # use the LogSumExp trick across n_clusters
        # get the component among n_clusters with the maximum value
        max_log = np.max(log_joint, axis=1, keepdims=True)
        print(f"max_log.shape = {max_log.shape}")
        # LogSumExp of the difference between the joint log probability
        # and the maximum value from n_clusters 
        # temp = np.exp(log_joint - max_log)
        # print(temp.shape)
        log_marginal_likelihood = max_log + np.log(np.sum(np.exp(log_joint - max_log), axis=1, keepdims=True))
        print(f"log_marginal_likelihood.shape = {log_marginal_likelihood.shape}")
        
        # get the responsibilities gamma 
        log_gamma = log_joint - log_marginal_likelihood
        gamma = np.exp(log_gamma)
        print(f"gamma.shape = {gamma.shape}")
        
        # # track convergence using total log likelihood 
        # total_log_likelihood = np.sum(log_marginal_likelihood)
        # print(f"total_log_likelihood = {total_log_likelihood}")
        # if prev_log_likelihood is not None and \
        # abs(total_log_likelihood - prev_log_likelihood) < tolerance:
        #     print(f"Converged at iteration ")
        # prev_log_likelihood = total_log_likelihood 

        # --- Maximization step ---
        # update parameters 
        # sum gamma over all training samples
        N_k = np.sum(gamma, axis = 0)
        print(f"N_k.shape = {N_k.shape}")
        # update prior weights, take average over all training samples 
        pi = N_k / n_samples
        print(f"pi.shape = {pi.shape}")

        # update weights (means) with smoothing 
        mu = (gamma.T @ X) / (N_k[:, np.newaxis] + smoothing)
        mu = np.clip(mu, 1e-10, 1 - 1e-10)

        diff = np.sum(np.abs(mu - prev_mu)) + np.sum(np.abs(pi - prev_pi))
        print(f"diff = {diff}")
        prev_mu = mu
        prev_pi = pi
        
        print()

    return mu, pi


    
# testing 
mu, pi = train_em_algorithm(images_train_thresholded)




X.shape = (60000, 784)
n_samples = 60000, n_dimensions = 784
pi.shape = (10,)
mu.shape = (10, 784)
Training
iteration 0
log_likelihood.shape = (60000, 10)
log_joint.shape = (60000, 10)
max_log.shape = (60000, 1)
log_marginal_likelihood.shape = (60000, 1)
gamma.shape = (60000, 10)
N_k.shape = (10,)
pi.shape = (10,)
diff = 2908.5121345491666

iteration 1
log_likelihood.shape = (60000, 10)
log_joint.shape = (60000, 10)
max_log.shape = (60000, 1)
log_marginal_likelihood.shape = (60000, 1)
gamma.shape = (60000, 10)
N_k.shape = (10,)
pi.shape = (10,)
diff = 279.4825990683686

iteration 2
log_likelihood.shape = (60000, 10)
log_joint.shape = (60000, 10)
max_log.shape = (60000, 1)
log_marginal_likelihood.shape = (60000, 1)
gamma.shape = (60000, 10)
N_k.shape = (10,)
pi.shape = (10,)
diff = 147.72886310803656

iteration 3
log_likelihood.shape = (60000, 10)
log_joint.shape = (60000, 10)
max_log.shape = (60000, 1)
log_marginal_likelihood.shape = (60000, 1)
gamma.shape = (60000, 10)
N_k.shape = (10

## predict

In [102]:
def predict_em_algorithm(X: np.array,
                         mu: np.array,
                        pi: np.array):
    # reshape dimensions from 28x28 to 784
    X = np.reshape(X, (X.shape[0], -1))
    
    print(f"Prediction")
    # smoothing = 1e-200
    # mu = np.clip(mu, smoothing, 1.0 - smoothing)
    # --- compute log likelihood --- 
    log_px_given_z = bernoulli_log_likelihood(X, mu)
    print(f"log_px_given_z.shape = {log_px_given_z.shape}")
    
    # --- get the joint probability ---
    log_joint = log_px_given_z + np.log(np.clip(pi, 1e-300, 1.0))
    print(f"log_joint.shape = {log_joint.shape}")
    
    # get the component among n_clusters with the maximum value
    max_log = np.max(log_joint, axis=1, keepdims = True)
    print(f"max_log.shape = {max_log.shape}")

    # marginal log likelihood
    # LogSumExp of the difference between the joint log probability
    # and the maximum value from n_clusters 
    log_px = max_log + np.exp(np.sum(np.exp(log_joint - max_log), axis=1, keepdims=True))
    print(f"log_px.shape = {log_px.shape}")

    # get the prediction probabilities from the exponential of 
    # the difference between the joint probability logs and the 
    # marginal log likelihoods
    predict_probs = np.exp(log_joint - log_px)
    print(f"predict_probs.shape = {predict_probs.shape}")
    
    predictions = np.argmax(predict_probs, axis=1)
    print(f"predictions = {predictions}")
    print(f"predictions.shape = {predictions.shape}")
    print(f"{np.unique(predictions)}")
    return predictions
    
predictions = predict_em_algorithm(images_train_thresholded, mu, pi)
# predictions
# labels_train

Prediction
log_px_given_z.shape = (60000, 10)
log_joint.shape = (60000, 10)
max_log.shape = (60000, 1)
log_px.shape = (60000, 1)
predict_probs.shape = (60000, 10)
predictions = [9 1 0 ... 9 6 3]
predictions.shape = (60000,)
[0 1 2 3 4 5 6 7 8 9]


In [107]:
print(f"predictions.shape = {predictions.shape}")
print(f"labels_train.shape = {labels_train.shape}")
for i in range(10):
    print(predictions[predictions == i].shape)
print()
for i in range(10):
    print(labels_train[labels_train == i].shape)

cluster_to_label = [0] * 10
for i in range(10):
    # get each cluster
    uniques, unique_counts = np.unique(labels_train[predictions == i], return_counts=True)
    cluster_to_label[i] = uniques[np.argmax(unique_counts)]
    
cluster_to_label
    # print(labels_train[labels_train == i].shape)

predictions.shape = (60000,)
labels_train.shape = (60000,)
(5318,)
(4593,)
(7197,)
(5780,)
(5656,)
(6636,)
(4767,)
(6731,)
(6209,)
(7113,)

(5923,)
(6742,)
(5958,)
(6131,)
(5842,)
(5421,)
(5918,)
(6265,)
(5851,)
(5949,)


[np.uint8(4),
 np.uint8(0),
 np.uint8(1),
 np.uint8(5),
 np.uint8(2),
 np.uint8(9),
 np.uint8(6),
 np.uint8(3),
 np.uint8(7),
 np.uint8(8)]

# Print digit class imaginations

In [108]:
def print_imaginations(mu: np.array):
    print(f"mu.shape = {mu.shape}")
    print(np.min(mu, axis=1), np.max(mu, axis=1))
    mu = np.reshape(mu, (mu.shape[0], 28, 28))
    mu = (mu >= 0.5).astype(int)
    
    for i in range(mu.shape[0]):
        image = mu[i]
        for row in image:
            # str1 = ""
            # for val in row:
            #     if val == 0:
            #         str1 += "  "
            #     else:
            #         str1 += "1"
            # print(str1)
            print(" ".join(str(val) for val in row))
        
print_imaginations(mu)

mu.shape = (10, 784)
[1.e-10 1.e-10 1.e-10 1.e-10 1.e-10 1.e-10 1.e-10 1.e-10 1.e-10 1.e-10] [0.81407831 0.83424711 0.95206673 0.66195234 0.7432476  0.86407478
 0.87047864 0.82045781 0.90228192 0.85172835]
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0
0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0
0 0 0 0 0 

# print confusion matrix

In [109]:
cluster_to_label

[np.uint8(4),
 np.uint8(0),
 np.uint8(1),
 np.uint8(5),
 np.uint8(2),
 np.uint8(9),
 np.uint8(6),
 np.uint8(3),
 np.uint8(7),
 np.uint8(8)]

In [115]:
predictions2 = np.zeros_like(predictions)
for i in range(len(predictions)):
    predictions2[i] = cluster_to_label[predictions[i]]
predictions2

array([8, 0, 4, ..., 8, 6, 5], shape=(60000,))

In [123]:
predictions[:20]

array([9, 1, 0, 2, 5, 4, 2, 9, 2, 3, 9, 3, 7, 6, 2, 8, 4, 3, 6, 8])

In [124]:
predictions2[:20]

array([8, 0, 4, 1, 9, 2, 1, 8, 1, 5, 8, 5, 3, 6, 1, 7, 2, 5, 6, 7])

In [125]:
labels_train[:20]

array([5, 0, 4, 1, 9, 2, 1, 3, 1, 4, 3, 5, 3, 6, 1, 7, 2, 8, 6, 9],
      dtype=uint8)

In [130]:

def print_metrics(y_pred: np.array,
                y_ground: np.array,
                 class_num: int,
                title: str = "title"):
    print(f"{title}")
    print("----------------------------------------------------------------------")
    print(f"{np.sum(y_pred == class_num)}, {np.sum(y_ground == class_num)}")
    tp = np.sum((y_pred == class_num) & (y_ground == class_num))
    tn = np.sum((y_pred != class_num) & (y_ground != class_num))
    fp = np.sum((y_pred == class_num) & (y_ground != class_num))
    fn = np.sum((y_pred != class_num) & (y_ground == class_num))
    
    # print confusion matrix
    print(f"Confusion Matrix {class_num}:\n")
    print(f"\t\tPredict 0\tPredict 1")
    print(f"Real 0\t\t{tn}\t\t{fp}")
    print(f"Real 1\t\t{fn}\t\t{tp}")
    
    # print sensitivity and specificity
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    print(f"Sensitivity (successfully predict cluster 1): {sensitivity}")
    print(f"Specificity (successfully predict cluster 2): {specificity}")
    
# print_metrics(y_preds, y_arr)
for i in range(10):
    print_metrics(predictions2, labels_train, i)

title
----------------------------------------------------------------------
4593, 5923
Confusion Matrix 0:

		Predict 0	Predict 1
Real 0		53853		224
Real 1		1554		4369
Sensitivity (successfully predict cluster 1): 0.7376329562721594
Specificity (successfully predict cluster 2): 0.995857758381567
title
----------------------------------------------------------------------
7197, 6742
Confusion Matrix 1:

		Predict 0	Predict 1
Real 0		52242		1016
Real 1		561		6181
Sensitivity (successfully predict cluster 1): 0.9167902699495699
Specificity (successfully predict cluster 2): 0.9809230538135116
title
----------------------------------------------------------------------
5656, 5958
Confusion Matrix 2:

		Predict 0	Predict 1
Real 0		52845		1197
Real 1		1499		4459
Sensitivity (successfully predict cluster 1): 0.7484055052030882
Specificity (successfully predict cluster 2): 0.9778505606750305
title
----------------------------------------------------------------------
6731, 6131
Confusion Matri